In [1]:
##### Logistic Regression #####
# Inspired by the algorithm Linear Regression, adding the usage of the sigmoid function
# sigmoid(x) = 1/(1 + e^-z), where z = x * w.T + b
# How does it work? It assumes once again a linear relationship between I/O, except it chooses a threshold
# Let's say we are trying to solve a decision problem ( a computational problem whose output is in the form of Y/N, 1/0, T/F etc. )
# Since it's binary, the threshold will be 0.5, i.e. probabilities > 0.5 will mean choice A, whereas <= 0.5 will mean choice B
# To compute loss it implements binary cross entropy, loss = -1/n * (y * logp + (1-y) * log(1-p))
# As for precision parameters, it is the first time we encounter the CONFUSION MATRIX
# For a decision problem it will be a 2x2 matrix : True Pos | False Neg
#                                                 False Pos | True Neg
#
# It is crucial in order to compute other scores, such as F1 score, precision, accuracy, recall.
import numpy as np
import pandas as pd
import kagglehub

dataset = None
x = None
targets = None
model = None

def load_data():
    """
    Data preprocessing function. Includes standardisation of data.
    """
    global x, targets
    global dataset
    
    dataset = pd.read_csv("/kaggle/input/datasets/mathchi/diabetes-data-set/diabetes.csv")
    dataset = dataset.drop(columns=["SkinThickness"])
    targets = np.array(list(dataset["Outcome"]))
    
    x = [None] * 7
    
    for i,col in enumerate(dataset):
        if i == 7:
            break
        x[i] = dataset[col]
    
    x = np.array(x)
    x_m = x.mean(axis = 1, keepdims = True)
    x_std = x.std(axis = 1, keepdims = True)
    x = (x - x_m) / x_std

def sigmoid(input): # we define the sigmoid function, input is w*x + b
    return 1/(1 + np.exp(-input))

def compute_loss(pred, target): # binary cross-entropy
    return -np.mean(target * np.log(pred) + (1-target) * np.log(1 - pred))
    
class LogisticRegression():
    """
    Logistic Regression model. Similar to Linear Regression, except it computes probabilities.
    """
    def __init__(self):
        self.loss = None
        self.maxiters = 200000
        self.w = None
        self.bias = None
        self.lr = 1e-1

    def fit(self, x, y):
        self.w = np.random.normal( size=(7,1) )
        self.bias = 0
        m = x.shape[1]
        
        for _ in range(self.maxiters):
            y_pred = sigmoid(self.w.T @ x + self.bias)
            self.loss = compute_loss(y_pred, y)
            
            self.w = self.w - self.lr * (x @ (y_pred - y).T )* 1/m
            self.bias = self.bias - self.lr * np.mean(y_pred - y)

    def predict(self, x):
        return sigmoid(self.w.T @ x + self.bias)

load_data()
targets = targets.reshape(1, -1)
print(targets.shape)
print(x.shape)

model = LogisticRegression()

# splits into train test split
train_y = targets[:, :int(targets.shape[1] * 0.8)]
test_y = targets[:, int(targets.shape[1] * 0.8):]

train_x = x[:, :int(x.shape[1] * 0.8)]
test_x = x[:, int(x.shape[1] * 0.8):]

model.fit(train_x, train_y)
train_pred = model.predict(train_x)
test_pred = model.predict(test_x)

train_pred_class = (train_pred > 0.5).astype(int)
test_pred_class = (test_pred > 0.5).astype(int)

# instead of using sklearn, i tried my best to compute the scores manually
def print_metrics(y_true, y_pred_prob, y_pred_class, name=""):
    y_true = y_true.ravel()
    y_pred_class = y_pred_class.ravel()
    y_pred_prob = y_pred_prob.ravel()
    
    accuracy = np.mean(y_true == y_pred_class)
    tp = np.sum((y_true == 1) & (y_pred_class == 1))
    fp = np.sum((y_true == 0) & (y_pred_class == 1))
    fn = np.sum((y_true == 1) & (y_pred_class == 0))
    
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
    
    print(name)
    print(f"Accuracy : {accuracy:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall   : {recall:.4f}")
    print(f"F1 Score : {f1:.4f}")

print_metrics(train_y, train_pred, train_pred_class, "TRAIN")
print_metrics(test_y, test_pred, test_pred_class, "TEST")

print(f"\nDiabetes in tests: {test_y.mean():.2%}")


##### Use cases #####
# -> mainly decision problems
# -> data where there is some sort of linear relationship

(1, 768)
(7, 768)

=== TRAIN ===
Accuracy : 0.7866
Precision: 0.7440
Recall   : 0.5869
F1 Score : 0.6562
Final Loss: 0.4686

=== TEST ===
Accuracy : 0.7662
Precision: 0.7317
Recall   : 0.5455
F1 Score : 0.6250
Final Loss: 0.4686

Procent diabet în test: 35.71%
